# Vision Transformer (ViT) from Scratch

This notebook walks through how a Vision Transformer works, step by step.
We will build the model from scratch, understand the math behind each piece,
and then train it on CIFAR-10 to classify 10 different types of images.

No prior knowledge of transformers is assumed. We start from the basics.

---

## The Big Idea

Before transformers, most image classifiers used Convolutional Neural Networks (CNNs).
CNNs work by sliding small filters across the image to detect local features like edges,
corners, and textures.

The Vision Transformer takes a completely different approach:

1. Cut the image into small square patches (like puzzle pieces).
2. Flatten each patch into a 1D vector.
3. Feed that sequence of vectors into a Transformer, the same architecture used in language models.
4. Let the Transformer figure out which patches are related to each other using attention.

The key insight is that the Transformer does not care whether the input is words or image patches.
It just sees a sequence of vectors and learns to relate them to each other.

---

## Paper Reference

The original paper is: *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*
(Dosovitskiy et al., 2020). The core idea fits in one sentence: treat image patches like words.

## Step 1: Cutting an Image into Patches

Suppose we have an image that is 32 pixels tall and 32 pixels wide, with 3 color channels (RGB).
We choose a patch size of 8 pixels.

That gives us:

```
number of patches = (image_height / patch_height) * (image_width / patch_width)
                  = (32 / 8) * (32 / 8)
                  = 4 * 4
                  = 16 patches
```

Each patch covers an 8x8 region of the image. When we flatten that patch, we get:

```
patch_dim = patch_height * patch_width * channels
          = 8 * 8 * 3
          = 192 values
```

So the entire image becomes a sequence of 16 vectors, each of length 192.
This is the input to our Transformer.

In the code we use `einops.Rearrange` to do this reshaping in one line:

```
b c (h p1) (w p2) -> b (h w) (p1 p2 c)
```

Reading left to right: batch, channels, (grid_rows * patch_height), (grid_cols * patch_width)
becomes: batch, (grid_rows * grid_cols), (patch_height * patch_width * channels).

After that, a linear layer projects each 192-dim patch vector into a `dim`-dimensional embedding.

## Step 2: Positional Encoding

The Transformer processes all patches at once with no built-in sense of order or position.
If we shuffle the patches, the model has no way to know they moved.
We fix this by adding positional information to each patch embedding.

There are two common approaches:

### Learnable Positions (used in full ViT)

We create a learnable parameter tensor of shape `(num_patches, dim)` and add it to the patch embeddings.
The model learns the best position encoding during training.

```python
self.pos_embedding = nn.Parameter(torch.randn(num_patches, dim))
```

### Sinusoidal Positions (used in SimpleViT)

We compute a fixed encoding using sine and cosine functions at different frequencies.
For a 2D grid of patches we encode the row and column positions separately.

For each patch at grid position (row, col):

```
frequency[i] = 1 / (10000 ^ (i / (dim/4)))

encoding = [ sin(row * freq[0]), sin(row * freq[1]), ...,
             cos(row * freq[0]), cos(row * freq[1]), ...,
             sin(col * freq[0]), sin(col * freq[1]), ...,
             cos(col * freq[0]), cos(col * freq[1]), ... ]
```

The sine and cosine at different frequencies create unique fingerprints for each position.
Low frequencies capture coarse position (is this patch near the top or bottom?)
and high frequencies capture fine position (is this in column 3 or 4?).

This encoding is fixed and does not need training. We just compute it once and add it.

## Step 3: Self-Attention

Self-attention is the core operation that lets each patch look at every other patch
and decide how much to borrow from it.

### The Query, Key, Value Framework

Think of it like a search engine. Each patch (token) has:
- A **Query** (Q): what this patch is looking for
- A **Key** (K): what this patch is advertising about itself
- A **Value** (V): what this patch actually shares when selected

We compute Q, K, V by multiplying the input by three separate weight matrices:

```
Q = X * W_Q     shape: (batch, num_patches, head_dim)
K = X * W_K     shape: (batch, num_patches, head_dim)
V = X * W_V     shape: (batch, num_patches, head_dim)
```

In code we do this efficiently with a single linear layer that outputs 3 * head_dim,
then split it into Q, K, V.

### Computing Attention Scores

To find how much patch i should attend to patch j, we compute the dot product of Q[i] and K[j].
A high dot product means the query and key are similar, so patch i will borrow a lot from patch j.

```
scores = Q * K^T / sqrt(head_dim)
```

We divide by sqrt(head_dim) to prevent the dot products from growing too large.
If the values are too large, the softmax becomes very sharp (near one-hot) and gradients vanish.

For head_dim = 64:
```
scale = 1 / sqrt(64) = 1 / 8 = 0.125
```

### Turning Scores into Weights

We pass the scores through softmax so they sum to 1 and can be interpreted as probabilities:

```
attention_weights = softmax(scores)     shape: (batch, num_patches, num_patches)
```

### Getting the Output

We use the attention weights to take a weighted average of the values:

```
output = attention_weights * V          shape: (batch, num_patches, head_dim)
```

Each patch now holds a mix of information from all other patches, weighted by relevance.

### Multi-Head Attention

Instead of one set of Q, K, V, we run multiple heads in parallel.
Each head learns to look for different kinds of relationships.
One head might focus on spatial closeness, another on color similarity, another on semantic meaning.

With num_heads = 8 and head_dim = 64, the inner dimension is 8 * 64 = 512.
We split the 512-dim projections into 8 groups of 64 and run attention independently on each.
Then we concatenate the results and project back to `dim`.

```
for each head h:
    scores_h = Q_h * K_h^T / sqrt(head_dim)
    weights_h = softmax(scores_h)
    out_h = weights_h * V_h

output = concat(out_0, out_1, ..., out_7) * W_out
```

## Step 4: Feed-Forward Network

After attention, each token is passed through a small MLP independently.
This is often called the position-wise feed-forward network.

```
FFN(x) = Linear(GELU(Linear(x)))
```

The first linear expands the dimension (usually by 4x), GELU adds non-linearity,
and the second linear contracts back to the original dim.

GELU is similar to ReLU but smoother. For a value x:
```
GELU(x) = x * Phi(x)
```
where Phi is the cumulative distribution function of the normal distribution.
This keeps most of the signal but smoothly zeros out negative values.

The feed-forward layer lets each token transform its own representation after seeing
information from other tokens through attention.

## Step 5: The Transformer Block

One Transformer block combines attention and feed-forward with residual connections and layer norm:

```
x = x + Attention(LayerNorm(x))
x = x + FeedForward(LayerNorm(x))
```

The residual connection (adding x back) means the block only needs to learn
the correction to apply, not the full transformation. This makes training much more stable
and prevents gradients from vanishing in deep networks.

LayerNorm normalizes the values along the feature dimension:

```
LayerNorm(x) = (x - mean(x)) / (std(x) + epsilon) * gamma + beta
```

This keeps the values in a healthy range before each sub-layer.
Note that we normalize before the operation (Pre-LN), not after (Post-LN).
Pre-LN tends to be more stable during training.

We stack multiple blocks (called depth in the code) to let the model learn richer representations.

## Step 6: The CLS Token and Classification Head

After the Transformer processes all patch tokens, we need one vector to classify the image.
There are two ways to get that vector:

### Option A: CLS Token (Classification Token)

We prepend a special learnable token to the sequence before the Transformer.
The model learns to use this token to collect and summarize information from all patches.
At the end, we take just the CLS token output and feed it to a linear classifier.

This is borrowed directly from BERT in NLP. The CLS token has no image content of its own,
so the model is forced to fill it with a useful global summary via attention.

### Option B: Mean Pooling

Alternatively, we average all patch token outputs:

```
output = mean(transformer_output, dim=1)
```

SimpleViT uses this approach because it removes the need for the CLS token
and works just as well in practice.

### The Classifier

The final step is a single linear layer that maps the `dim`-dimensional vector to `num_classes`:

```
logits = Linear(dim, num_classes)(pooled_output)
```

We get one score per class. The highest score is the predicted class.

---

## Let's Build It

Now we will implement all the pieces above and train a ViT on CIFAR-10.

CIFAR-10 has 60,000 images across 10 classes: airplane, car, bird, cat, deer,
dog, frog, horse, ship, and truck. Each image is 32x32 pixels with 3 channels.

We will use patches of size 8x8, giving us a 4x4 = 16 patch grid.

In [ ]:
# install dependencies
!pip install torch torchvision einops tqdm

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from einops import rearrange, repeat
from einops.layers.torch import Rearrange

from tqdm import tqdm

print('torch version:', torch.__version__)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using device:', device)

## Load CIFAR-10

CIFAR-10 images are 32x32 with 3 channels. We normalize them using the dataset mean and std,
which are well-known values for CIFAR-10:
- mean per channel: (0.4914, 0.4822, 0.4465)
- std per channel: (0.2023, 0.1994, 0.2010)

Normalization puts each channel close to zero mean and unit variance,
which helps the model learn faster and more stably.

For training we add random horizontal flipping and random cropping as data augmentation.
This makes the model see more varied examples and helps it generalize.

In [ ]:
# cifar10 mean and std
cifar_mean = (0.4914, 0.4822, 0.4465)
cifar_std = (0.2023, 0.1994, 0.2010)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

class_names = ['airplane', 'car', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print('train batches:', len(train_loader))
print('test batches:', len(test_loader))

## Build the Feed-Forward Block

This is a simple two-layer MLP. The hidden dimension is usually 4x the input dimension.
We apply LayerNorm first, then expand, activate, then contract back.

The expansion step lets the model have more capacity for mixing features.
Think of it like doing rough draft work in a large space before cleaning up.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.linear1 = nn.Linear(dim, hidden_dim)
        self.activation = nn.GELU()
        self.linear2 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        x = self.norm(x)
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        return x

## Build Multi-Head Attention

Let's walk through the shapes step by step for a single forward pass.

Suppose:
- batch = 4
- num_patches = 16  (4x4 grid from a 32x32 image with 8x8 patches)
- dim = 256
- num_heads = 8
- head_dim = 32
- inner_dim = 8 * 32 = 256

```
x shape:             (4, 16, 256)
after norm:          (4, 16, 256)
after to_qkv:        (4, 16, 768)   <- 256 * 3
after chunk:         q, k, v each (4, 16, 256)
after rearrange:     q, k, v each (4, 8, 16, 32)  <- b h n d
scores:              (4, 8, 16, 16)  <- each head compares all patches
attn_weights:        (4, 8, 16, 16)  <- after softmax
out:                 (4, 8, 16, 32)
after rearrange:     (4, 16, 256)   <- back to original shape
after to_out:        (4, 16, 256)
```

Each of the 16 patches now holds a mix of information from all other patches.

In [ ]:
class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, head_dim=32):
        super().__init__()
        inner_dim = head_dim * num_heads
        self.num_heads = num_heads
        # prevent large dot products
        self.scale = head_dim ** -0.5
        self.norm = nn.LayerNorm(dim)
        self.softmax = nn.Softmax(dim=-1)

        # one linear produces q, k, v together
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.to_out = nn.Linear(inner_dim, dim, bias=False)

    def forward(self, x):
        x = self.norm(x)

        # split last dim into q, k, v
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(
            lambda t: rearrange(t, 'b n (h d) -> b h n d', h=self.num_heads),
            qkv
        )

        scores = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        attn_weights = self.softmax(scores)

        out = torch.matmul(attn_weights, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

## Build the Transformer

The Transformer is just a stack of identical blocks. Each block has attention and feed-forward,
both with residual connections.

At the end we apply a final LayerNorm. This is standard practice to stabilize the output
before the classification head reads it.

In [ ]:
class Transformer(nn.Module):
    def __init__(self, dim, depth, num_heads, head_dim, mlp_dim):
        super().__init__()
        self.final_norm = nn.LayerNorm(dim)
        self.layers = nn.ModuleList([])

        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                Attention(dim, num_heads, head_dim),
                FeedForward(dim, mlp_dim)
            ]))

    def forward(self, x):
        for attn, ff in self.layers:
            # residual connections
            x = attn(x) + x
            x = ff(x) + x
        return self.final_norm(x)

## Build the Full ViT

Now we assemble all the pieces. Here is the full forward pass in plain English:

1. Input image: shape (batch, 3, 32, 32)
2. Cut into 16 patches: shape (batch, 16, 192)  <- 8*8*3 per patch
3. Project each patch to dim=256: shape (batch, 16, 256)
4. Add positional embedding so the model knows where each patch is
5. Run through 6 Transformer blocks
6. Average all 16 token outputs: shape (batch, 256)
7. Pass through a linear layer to get 10 class scores: shape (batch, 10)

We use sinusoidal positional encoding (no extra parameters to learn) and mean pooling
(no CLS token needed). This is the SimpleViT approach.

The sinusoidal encoding uses the same formula as the original Transformer paper.
For a 2D image grid we encode row and column positions separately using 4 groups of frequencies:
sin and cos for rows, sin and cos for columns. Each group takes up dim/4 slots.

In [ ]:
def build_sincos_embedding(num_rows, num_cols, dim):
    assert dim % 4 == 0, 'dim must be divisible by 4'

    row_ids = torch.arange(num_rows)
    col_ids = torch.arange(num_cols)

    grid_rows, grid_cols = torch.meshgrid(row_ids, col_ids, indexing='ij')
    grid_rows = grid_rows.flatten().float()
    grid_cols = grid_cols.flatten().float()

    # compute frequencies
    freq_count = dim // 4
    freq_bands = torch.arange(freq_count).float() / max(freq_count - 1, 1)
    freq_bands = 1.0 / (10000.0 ** freq_bands)

    row_enc = grid_rows[:, None] * freq_bands[None, :]
    col_enc = grid_cols[:, None] * freq_bands[None, :]

    # concat four groups
    pos_embed = torch.cat([
        row_enc.sin(),
        row_enc.cos(),
        col_enc.sin(),
        col_enc.cos(),
    ], dim=1)

    return pos_embed

In [ ]:
class SimpleViT(nn.Module):
    def __init__(
        self,
        image_size,
        patch_size,
        num_classes,
        dim,
        depth,
        num_heads,
        mlp_dim,
        channels=3,
        head_dim=32
    ):
        super().__init__()

        # split image size and patch size
        if isinstance(image_size, int):
            image_h = image_w = image_size
        else:
            image_h, image_w = image_size

        if isinstance(patch_size, int):
            patch_h = patch_w = patch_size
        else:
            patch_h, patch_w = patch_size

        assert image_h % patch_h == 0 and image_w % patch_w == 0, \
            'image size must divide evenly by patch size'

        patch_dim = channels * patch_h * patch_w
        num_patch_rows = image_h // patch_h
        num_patch_cols = image_w // patch_w

        # cut image into patches and project
        self.patch_embed = nn.Sequential(
            Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=patch_h, p2=patch_w),
            nn.LayerNorm(patch_dim),
            nn.Linear(patch_dim, dim),
            nn.LayerNorm(dim),
        )

        # register position encoding as a buffer (not a parameter)
        self.register_buffer(
            'pos_embedding',
            build_sincos_embedding(num_patch_rows, num_patch_cols, dim)
        )

        self.transformer = Transformer(dim, depth, num_heads, head_dim, mlp_dim)
        self.classifier = nn.Linear(dim, num_classes)

    def forward(self, img):
        x = self.patch_embed(img)
        # add position info
        x = x + self.pos_embedding
        x = self.transformer(x)
        # average all tokens
        x = x.mean(dim=1)
        return self.classifier(x)

## Model Hyperparameters

For CIFAR-10 we use a small model because the images are small (32x32).
Larger models would overfit without more data.

- image_size = 32: each image is 32x32 pixels
- patch_size = 8: each patch is 8x8 pixels, giving us 16 patches per image
- dim = 256: each patch gets embedded as a 256-dimensional vector
- depth = 6: we stack 6 Transformer blocks
- num_heads = 8: 8 attention heads, each looking at a 32-dim subspace
- mlp_dim = 512: the feed-forward hidden layer is 2x the embedding dim
- num_classes = 10: one output score per CIFAR-10 class

Total parameters for this model: roughly 3-4 million. Small enough to train on a CPU
in reasonable time, though a GPU is much faster.

In [ ]:
model = SimpleViT(
    image_size=32,
    patch_size=8,
    num_classes=10,
    dim=256,
    depth=6,
    num_heads=8,
    mlp_dim=512,
    channels=3,
    head_dim=32
).to(device)

# count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'total trainable parameters: {total_params:,}')

# quick shape check
dummy = torch.randn(4, 3, 32, 32).to(device)
out = model(dummy)
print(f'input shape:  {dummy.shape}')
print(f'output shape: {out.shape}')

## Training Setup

We use the Adam optimizer with weight decay, often called AdamW.
Weight decay adds a small penalty for large weights, which helps prevent overfitting.

The learning rate starts at 3e-4 and decays by 10x after epoch 15.
A warmup schedule or cosine decay would be better for larger models,
but step decay is easy to understand and works reasonably here.

CrossEntropyLoss combines log-softmax and negative log likelihood in one step.
For each example it penalizes the model proportionally to how wrong it was.

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

num_epochs = 20
learning_rate = 3e-4

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = StepLR(optimizer, step_size=15, gamma=0.1)

## Training Loop

The training loop does the same thing every epoch:

1. For each batch, run the model forward to get predictions (logits).
2. Compute the loss between predictions and ground truth labels.
3. Run backward to compute gradients.
4. Update the model weights using the optimizer.
5. After all training batches, evaluate on the test set without gradients.

We track accuracy as the fraction of examples where the predicted class (highest logit)
matches the true label.

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # count correct predictions
        predicted = logits.argmax(dim=1)
        total_correct += (predicted == labels).sum().item()
        total_samples += labels.size(0)
        total_loss += loss.item() * labels.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy


def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = loss_fn(logits, labels)

            predicted = logits.argmax(dim=1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)
            total_loss += loss.item() * labels.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy

In [ ]:
train_losses = []
train_accs = []
test_losses = []
test_accs = []

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)
    scheduler.step()

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    test_losses.append(test_loss)
    test_accs.append(test_acc)

    print(
        f'epoch {epoch:02d} '
        f'train loss: {train_loss:.4f} train acc: {train_acc:.4f} '
        f'test loss: {test_loss:.4f} test acc: {test_acc:.4f}'
    )

## Plot the Training Curves

A training curve shows how the loss and accuracy change over time.

What to look for:
- Both train and test loss should decrease together. Good.
- If train loss keeps dropping but test loss starts rising, the model is overfitting.
- If both losses stop decreasing early, you might need a higher learning rate or more capacity.

For a ViT on CIFAR-10 without pretraining, expect around 65-75% test accuracy
with this small model and 20 epochs. Larger models or longer training would help.

In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, num_epochs + 1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_losses, label='train')
axes[0].plot(epochs, test_losses, label='test')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross Entropy Loss')
axes[0].legend()

axes[1].plot(epochs, train_accs, label='train')
axes[1].plot(epochs, test_accs, label='test')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'final test accuracy: {test_accs[-1]:.4f}')

## Inspect Some Predictions

Let's look at how the model does on a batch of test images.
We will show each image with its true label and what the model guessed.

To display the images properly we reverse the normalization so they look like normal photos again.
This is just for visualization and does not affect the model.

In [ ]:
import numpy as np

model.eval()

# grab one batch
images, labels = next(iter(test_loader))
images_gpu = images.to(device)
labels_gpu = labels.to(device)

with torch.no_grad():
    logits = model(images_gpu)

predicted_classes = logits.argmax(dim=1).cpu()

# show first 16 images
mean = torch.tensor(cifar_mean).view(3, 1, 1)
std = torch.tensor(cifar_std).view(3, 1, 1)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))

for idx, ax in enumerate(axes.ravel()):
    img = images[idx] * std + mean
    img = img.permute(1, 2, 0).numpy()
    img = img.clip(0, 1)

    true_label = class_names[labels[idx].item()]
    pred_label = class_names[predicted_classes[idx].item()]

    color = 'green' if true_label == pred_label else 'red'

    ax.imshow(img)
    ax.set_title(f'true: {true_label}\npred: {pred_label}', color=color, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Visualize Patch Splits

Let's see what the 4x4 patch grid actually looks like on a real image.
This makes it concrete how the image gets cut up before being fed to the Transformer.

Each cell in the grid is one token. The Transformer learns to relate these tokens to each other
through self-attention. A patch in the top-left corner can attend to a patch in the bottom-right
without any intermediate steps, which is a big advantage over CNNs.

In [ ]:
# show one image with patch grid overlay
sample_img = images[0] * std + mean
sample_img = sample_img.permute(1, 2, 0).numpy().clip(0, 1)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(sample_img, interpolation='nearest')

patch_size = 8
image_size = 32

# draw grid lines at patch boundaries
for i in range(0, image_size + 1, patch_size):
    ax.axhline(i - 0.5, color='white', linewidth=1.5)
    ax.axvline(i - 0.5, color='white', linewidth=1.5)

ax.set_title(f'class: {class_names[labels[0].item()]}')
ax.axis('off')
plt.tight_layout()
plt.show()

num_patches = (image_size // patch_size) ** 2
patch_dim = patch_size * patch_size * 3
print(f'patch grid: {image_size // patch_size} x {image_size // patch_size} = {num_patches} patches')
print(f'each patch flattened: {patch_size} * {patch_size} * 3 = {patch_dim} values')

## Summary

Here is what we built, in order:

1. **Patch Embedding**: Cut the 32x32 image into 16 patches of 8x8, flatten each patch,
   and project them to 256 dimensions with a linear layer.

2. **Positional Encoding**: Add sinusoidal encodings based on the row and column index
   of each patch, so the model knows where each patch came from.

3. **Multi-Head Attention**: Let each patch attend to every other patch.
   We compute similarity scores via Q*K/sqrt(d), apply softmax, then mix V by those weights.
   Multiple heads run in parallel, each learning different relationships.

4. **Feed-Forward**: After attention, each token independently goes through a small MLP
   to further transform its representation.

5. **Residual Connections**: Both attention and feed-forward are wrapped with
   x = x + block(LayerNorm(x)) to stabilize training and help gradients flow.

6. **Mean Pooling + Classifier**: After 6 Transformer blocks, average all 16 token outputs
   and pass through a linear layer to get 10 class scores.

The strength of ViT is that it has a global receptive field from the very first block.
Any patch can directly attend to any other patch, regardless of distance.
CNNs build up global context gradually through many layers of local convolutions.

The weakness is that ViT needs a lot of data or pretraining to reach its full potential.
On small datasets like CIFAR-10 without pretraining, CNNs usually outperform ViT.
But at scale, ViT often wins.